# Finaler Datensatz
**Zusammenführung aller Flags + Archetyp-Kombinationen für das Testgebiet**

Die gemeinsame Arbeitsdatei `nrw_buildings.parquet` enthält bereits alle
Cluster-, Dach- und Spitzenleistungs-Spalten sowie Rasenfläche und das erste
Geothermie-Flag (angereichert von 02, 03 und 05). Dieses Notebook ergänzt die
Schritte, die räumliche Zusatzdaten brauchen:

1. **Zweites Geothermie-Flag** — Wärmeleitfähigkeit (ews100-Raster) → `capacity`
2. **Drittes Geothermie-Flag** — Wasserschutzgebiete → `geothermal_potential`
3. **Kombiniertes Geothermie-Flag** → `geothermal_activated_1/2/3`
4. **Gasverfügbarkeit** — Zensus 2021 → `gas_available`
5. **Luft-Wärmepumpen-Obstruction** → `air_hp_outside_1/2/3`
6. Finale Spaltenauswahl → `nrw_results/final_dataset.parquet`
7. Testgebiet Ochtrup → einzigartige Archetyp-Kombinationen für das SESMG

## 1. Konfiguration

In [ ]:
import json
from pathlib import Path
from shapely.geometry import Polygon
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)   # alle Spalten anzeigen

BASE_DIR      = "."
PROJEKT_ROOT  = Path.cwd()
FILE_GEBAEUDE = os.path.join(BASE_DIR, "nrw_buildings.parquet")
OUTPUT_DIR    = os.path.join(BASE_DIR, "nrw_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Bereit")

## 2. Arbeitsdatei laden

`nrw_buildings.parquet` enthält bereits: Cluster-Repräsentanten und Dachdaten
(02), Spitzenleistung (03), Rasenfläche und erstes Geothermie-Flag (05).

In [ ]:
gdf = gpd.read_parquet(FILE_GEBAEUDE)
if gdf.geometry.name != "footprint":
    gdf = gdf.set_geometry("footprint")
print(f'Gebaeude: {len(gdf):,}  Spalten: {len(gdf.columns)}')
gdf.columns.tolist()

## 3. Zweites Geothermie-Flag — Wärmeleitfähigkeit

Der ews100-Raster (Erdwärmesonden-Potenzial, 100 m Zellen) liefert je Zelle eine
Wärmeleitfähigkeit. Buildings erhalten die `capacity` der Zelle, in der ihre
Grundfläche liegt.

In [ ]:
ews_100 = gpd.read_parquet(
    PROJEKT_ROOT / "geothermie" / "geothermie_waermeleitfaehigkeit" / "ews100_nrw_reduced.parquet"
)

# Right Join — jedes Gebaeude bleibt, auch ohne Ueberschneidung mit dem Raster
gdf = gpd.sjoin(ews_100, gdf, how="right", predicate="intersects", rsuffix="ews")
gdf.drop(columns=["index_left"], inplace=True, errors="ignore")

# ews-Zellen sind ein disjunktes Raster — hoechstens ein Treffer je Gebaeude.
# Falls doch mehrfach: ersten behalten, damit die Gebaeudezahl stabil bleibt.
gdf = gdf[~gdf.index.duplicated(keep="first")]

print(f'Gebaeude mit capacity-Wert : {gdf["capacity"].notna().sum():,}')
print(f'Gebaeude ohne (kein Raster) : {gdf["capacity"].isna().sum():,}')
print(gdf["capacity"].value_counts(dropna=False).to_string())

## 4. Drittes Geothermie-Flag — Wasserschutzgebiete

`geothermal_potential = True` heißt: Das Gebäude liegt **nicht** in einem
Wasserschutzgebiet (kein `wsg_zone`-Treffer).

In [ ]:
geothermie_gdf = gpd.read_file(
    str(PROJEKT_ROOT / "geothermie" / "wasserschutzgebiete" / "Wasserschutzgebiete-NRW.shp")
)
geothermie_gdf = geothermie_gdf.to_crs(epsg=25832)
geothermie_gdf_redu = geothermie_gdf.loc[:, ["wsg_zone", "geometry"]]

gdf = gpd.sjoin(geothermie_gdf_redu, gdf, how="right", predicate="intersects", rsuffix="geo")
gdf.drop(columns=["index_left"], inplace=True, errors="ignore")

# Schutzgebietszonen koennen sich ueberlappen (Zone II/III) — ersten Treffer
# behalten, damit die Gebaeudezahl stabil bleibt.
gdf = gdf[~gdf.index.duplicated(keep="first")]

# geothermal_potential = Gebaeude liegt NICHT in einem Wasserschutzgebiet
# (kein wsg_zone-Treffer; `.isna()` statt `== np.nan`, da NaN != NaN immer
# False ergibt).
gdf["geothermal_potential"] = gdf["wsg_zone"].isna()

print(f'Mit Potenzial (ausserhalb WSG)  : {gdf["geothermal_potential"].sum():,}')
print(f'Innerhalb Wasserschutzgebiet    : {(~gdf["geothermal_potential"]).sum():,}')

## 5. Kombiniertes Geothermie-Flag

Geothermie gilt als aktivierbar, wenn alle drei Flags erfüllt sind:
- `geothermal_potential` (kein Wasserschutzgebiet),
- `capacity == 2.5` (ausreichende Wärmeleitfähigkeit),
- `geothermal_feasible_X` (Rasenfläche reicht für die Sondenfläche, aus 05).

In [ ]:
# Vektorisierte Berechnung je Sanierungszustand.
for zustand in (1, 2, 3):
    gdf[f"geothermal_activated_{zustand}"] = (
        gdf["geothermal_potential"]
        & (gdf["capacity"] == 2.5)
        & gdf[f"geothermal_feasible_{zustand}"]
    )

for zustand in (1, 2, 3):
    n = gdf[f"geothermal_activated_{zustand}"].sum()
    print(f'  geothermal_activated_{zustand}: {n:,}  ({n/len(gdf)*100:.1f}%)')

## 6. Gasverfügbarkeit aus dem Zensus-Datensatz

Annahme: Wenn im Zensus-Gebiet Gasheizungen installiert sind, ist eine
Gasversorgung wahrscheinlich verfügbar.

In [ ]:
zensus_gdf = gpd.read_parquet(
    PROJEKT_ROOT / "zensus" / "zensus_daten_2021_polygone.parquet"
)
# Der Zensus hat eine eigene 'id'-Spalte (unabhaengig von der Gebaeude-ID) —
# hier vorher entfernen, damit die Gebaeude-ID 'id' heisst und nicht durch die
# Kollision umbenannt wird.
zensus_gdf = zensus_gdf.drop(columns=[c for c in ["id"] if c in zensus_gdf.columns])

gdf = gpd.sjoin(gdf, zensus_gdf, how="left", predicate="intersects", rsuffix="zensus")
# Zensuszellen sind disjunkt — hoechstens ein Treffer; falls doch mehrfach,
# ersten behalten.
gdf = gdf[~gdf.index.duplicated(keep="first")]

gdf["gas_available"] = gdf["Gas"].fillna(False).astype(int) >= 1

print(f'Gas verfuegbar    : {gdf["gas_available"].sum():,}')
print(f'Kein Gas / unklar : {(~gdf["gas_available"]).sum():,}')

## 7. Luft-Wärmepumpe — Installationshindernis

Vereinfachte Annahme: Eine Außenaufstellung (Außengerät) einer Luft-Wärmepumpe
gilt als möglich, wenn die Rasenfläche mehr als 15 m² beträgt. Bis die Logik
als eigener Schritt nach `06_nrw_luftwaermepumpe.ipynb` ausgelagert wird,
bleibt sie hier.

In [ ]:
gdf["air_hp_outside_1"] = gdf["lawn_area"] > 15
gdf["air_hp_outside_2"] = gdf["lawn_area"] > 15
gdf["air_hp_outside_3"] = gdf["lawn_area"] > 15

print(f'Aufstellung moeglich (>15 m2 Rasenflaeche): {gdf["air_hp_outside_1"].sum():,}')

## 8. Finale Spaltenauswahl und Speichern

Schreibt `nrw_results/final_dataset.parquet` — den SESMG-Input mit allen
Gebäude- und Flag-Spalten.

In [ ]:
FINALE_SPALTEN = [
    "footprint", "refurbishment_state", "construction_year", "area_m2",
    "Al_m2", "tabula_key", "elec_kwh", "heat_total_kwh", "nearest_city",
    "id", "size_class", "lawn_area",
    "geothermal_feasible_1", "geothermal_feasible_2", "geothermal_feasible_3",
    "capacity", "geothermal_potential",
    "geothermal_activated_1", "geothermal_activated_2", "geothermal_activated_3",
    "avg_tilt_deg", "area_north_m2", "area_south_m2", "roof_category",
    "has_pv_area", "area_eastwest_m2",
    "heat_cluster_1", "heat_cluster_2", "heat_cluster_3", "elec_cluster",
    "roof_cluster_eastwest", "roof_cluster_south",
    "gas_available",
    "air_hp_outside_1", "air_hp_outside_2", "air_hp_outside_3",
    "peak_heat_power_kw_1", "peak_heat_power_kw_2", "peak_heat_power_kw_3",
]

gdf = gdf[[c for c in FINALE_SPALTEN if c in gdf.columns]]
ausgabe_pfad = os.path.join(OUTPUT_DIR, "final_dataset.parquet")
gdf.to_parquet(ausgabe_pfad, index=False)
print(f"Gespeichert: {ausgabe_pfad}")
print(f'Zeilen: {len(gdf):,}  Spalten: {len(gdf.columns)}')

## 9. Gebäude im Testpolygon finden

`polygon` — das Ochtrup-Testgebiet, als WGS84 (lon/lat) in
`pilotprojects/ochtrup_polygon.json`, nach EPSG:25832 umprojiziert.

In [ ]:
with open(PROJEKT_ROOT / "pilotprojects" / "ochtrup_polygon.json",
          encoding="utf-8") as f:
    ochtrup_geojson = json.load(f)
polygon_wgs84 = Polygon(ochtrup_geojson["geometry"]["coordinates"][0])
polygon = gpd.GeoSeries([polygon_wgs84], crs="EPSG:4326").to_crs("EPSG:25832").iloc[0]

# Index-Positionen der Geometrien, die das Polygon schneiden
possible_matches_idx = gdf.sindex.query(polygon, predicate="intersects")

# Die eigentlichen Zeilen holen
test_buildings = gdf.iloc[possible_matches_idx]
test_buildings = test_buildings.drop_duplicates("id")
test_buildings.reset_index(drop=True, inplace=True)
test_buildings.index = range(1, len(test_buildings) + 1)
print(f'Gebaeude im Testgebiet: {len(test_buildings)}')
test_buildings

## 10. Einzigartige Kombinationen finden

Die Kombination besteht aus folgenden Variablen:
Wärmebedarf | Strombedarf | nächste Stadt | Sanierungszustand | Dachtyp |
Ost-West | Süd | Geothermie aktivierbar | Gas verfügbar | Luft-WP-Aufstellung

Die drei Zustands-Spalten (`heat_cluster`, `geothermal_activated`,
`air_hp_outside`) werden je Zeile anhand des Sanierungszustands aufgelöst
(Suffix _1/_2/_3).

In [ ]:
spalten_1 = [
    "size_class", "elec_cluster", "nearest_city", "refurbishment_state",
    "roof_category", "roof_cluster_eastwest", "roof_cluster_south", "gas_available"
]
spalten_2 = ["heat_cluster", "geothermal_activated", "air_hp_outside"]

# spalten_2 haengen vom refurbishment_state der jeweiligen Zeile ab (Suffix _1/_2/_3)
def zustandsabhaengiger_wert(zeile, spaltenname):
    zustand = int(zeile["refurbishment_state"])
    return zeile[f"{spaltenname}_{zustand}"]

for spalte in spalten_2:
    test_buildings[spalte] = test_buildings.apply(
        lambda zeile: zustandsabhaengiger_wert(zeile, spalte), axis=1)

alle_spalten = spalten_1 + spalten_2

test_buildings["combination_id"] = test_buildings[alle_spalten].astype(str).agg("|".join, axis=1)
unique_liste = list(test_buildings["combination_id"].unique())

kombis_pfad = os.path.join(OUTPUT_DIR, "combination_ids.txt")
with open(kombis_pfad, "w", encoding="utf-8") as f:
    # Spaltennamen schreiben
    f.write("|".join(alle_spalten) + "\n")
    # Jede einzigartige Kombination schreiben
    for kombi in unique_liste:
        f.write(f"{kombi}\n")

print(f'Einzigartige Kombinationen: {len(unique_liste)}')
print(f"Gespeichert: {kombis_pfad}")

## 11. Testgebiet visualisieren

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
test_buildings.plot(ax=ax)

for idx, row in test_buildings.iterrows():
    zentroid = row.footprint.centroid
    ax.annotate(
        text=str(idx),
        xy=(zentroid.x, zentroid.y),
        fontsize=7,
        color="red",
        ha="center",
        va="bottom"
    )

In [ ]:
test_buildings["west"] = test_buildings["area_eastwest_m2"] / 2
test_buildings[["avg_tilt_deg", "area_north_m2", "area_south_m2",
                "area_eastwest_m2", "west", "heat_cluster", "elec_cluster",
                "roof_cluster_eastwest", "roof_cluster_south", "gas_available",
                "air_hp_outside", "combination_id"]]